# AI Agent 起步 Notebook

Kernel 选 **Python (agent-lab)**。全部单元格离线可跑，不需要任何 API Key。

建议顺序：先跑通本 Notebook，再按 `README.md` 的六课路线逐个运行 `examples/` 里的脚本。

In [ ]:
import sys
import pathlib
import importlib.metadata as md

# 让 Notebook 能 import examples/ 下的公共模块（无论从哪个目录启动 Jupyter）
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "examples").exists())
sys.path.insert(0, str(root / "examples"))

print("项目根目录:", root)
for name in ["openai", "langchain", "langgraph", "mcp", "chromadb", "jupyterlab"]:
    print(f"  {name:12s} {md.version(name)}")

## 1. 不靠框架，看一次完整的 Agent 循环

`01_react_from_scratch.py` 里最重要的三个对象：`TOOLS`（工具）、`build_prompt`（短期记忆）、`agent`（循环）。
这里用离线 MockLLM 驱动，观察 Thought → Action → Observation → Final Answer。

In [ ]:
import importlib
react = importlib.import_module("01_react_from_scratch")

question = "先算 12 * (3 + 4)，再告诉我现在几点"
answer = react.agent(question, react.MockLLM(question))
print("\n最终答案：", answer)
print("\n可用工具：", list(react.TOOLS))

## 2. RAG：检索到底在算什么

改成 `alpha=0` 只看 BM25、`alpha=1` 只看向量，对比召回结果；再想想为什么「汽车」和「轿车」在这套哈希向量里不相似。

In [ ]:
from rag_utils import HybridRetriever, build_prompt

retriever = HybridRetriever()
query = "MCP 是干什么的？"
hits = retriever.search(query, k=2)
for i, (doc, score) in enumerate(hits, 1):
    print(f"[{i}] score={score:.3f}  {doc[:50]}...")
print("\n拼装后的 prompt:\n", build_prompt(query, hits))

## 下一步

1. 在终端跑 `python examples/05_mcp_client.py`，看工具如何被标准化复用。
2. 用 `python examples/mock_api_server.py` 起假端点，再跑 `02_function_calling.py` 和 `03_langgraph_agent.py`，观察真实的 tool_calls 往返。
3. 配好 `.env` 后把上面所有示例换成真实模型重跑一遍 —— 对比模型行为差异，是最好的一课。